# F1 Podium Predictor ML Pipeline
This notebook builds the machine learning pipeline for the Formula 1 podium prediction capstone project. It merges Ergast CSV data, engineers features, trains a RandomForest model and exports it for use in the Streamlit application.

## 1. Import Libraries
We'll bring in pandas, numpy, scikit-learn, joblib and any helpers we need.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

# for reproducibility
RANDOM_STATE = 42

## 2. Load CSV Files
Read results, races, drivers and constructors into dataframes.  Circuits is not strictly required for training but we can load it if needed later.

In [ ]:
base_path = "datasets/"

results = pd.read_csv(base_path + "results.csv")
races = pd.read_csv(base_path + "races.csv")
drivers = pd.read_csv(base_path + "drivers.csv")
constructors = pd.read_csv(base_path + "constructors.csv")
# circuits = pd.read_csv(base_path + "circuits.csv")

print("Loaded shapes:")
print(" results", results.shape)
print(" races", races.shape)
print(" drivers", drivers.shape)
print(" constructors", constructors.shape)

## 3. Merge DataFrames
Join the results dataframe with races (to pull year and circuitId), drivers, and constructors. This will give us all the information required to engineer our features.

In [ ]:
# merge year and circuitId from races
merged = results.merge(
    races[['raceId', 'year', 'circuitId']],
    on='raceId',
    how='left'
)

# merge driver information (we only need the id for modelling but include name for reference)
merged = merged.merge(
    drivers[['driverId']],
    on='driverId',
    how='left'
)

# merge constructor information
merged = merged.merge(
    constructors[['constructorId']],
    on='constructorId',
    how='left'
)

print("After merging, shape:", merged.shape)
merged.head()

## 4. Filter Recent Years
Keep only races that took place in 2010 or later to focus on the modern era of Formula 1.

In [ ]:
merged = merged[merged['year'] >= 2010].copy()
print("Filtered to >=2010, new shape:", merged.shape)

## 5. Define Target Variable
Create a `Podium` column: 1 if the finishing position (`positionOrder`) is 1, 2 or 3, otherwise 0.

In [ ]:
merged['Podium'] = np.where(merged['positionOrder'] <= 3, 1, 0)
merged[['positionOrder','Podium']].head(10)

## 6. Feature Engineering
Select the four features specified in the project brief: grid, circuitId, constructorId and driverId.  Convert any `
` strings to numeric and drop rows with missing values.

In [ ]:
# coerce columns to numeric ("\N" becomes NaN) and drop any rows with nulls
feature_cols = ['grid', 'circuitId', 'constructorId', 'driverId']
for col in feature_cols:
    merged[col] = pd.to_numeric(merged[col], errors='coerce')

X = merged[feature_cols].copy()
y = merged['Podium'].copy()

# drop NaNs that resulted from coercion
mask = X.notna().all(axis=1)
X = X[mask]
y = y[mask]

print("Features after cleaning", X.shape)
X.head()

## 7. Train Random Forest Classifier
Split the data into training and validation sets, then train a `RandomForestClassifier` on the features.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print("Training completed")

## 8. Evaluate Model Performance
Compute basic classification metrics on the held‑out test set: accuracy, precision, recall and F1‑score.

In [ ]:
y_pred = rf.predict(X_test)

dev_accuracy = accuracy_score(y_test, y_pred)
dev_precision = precision_score(y_test, y_pred)
dev_recall = recall_score(y_test, y_pred)
dev_f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {dev_accuracy:.3f}")
print(f"Precision: {dev_precision:.3f}")
print(f"Recall: {dev_recall:.3f}")
print(f"F1: {dev_f1:.3f}")

## 9. Save Trained Model
Export the fitted `RandomForestClassifier` to disk using joblib so it can be loaded by the Streamlit application.

In [ ]:
joblib.dump(rf, 'f1_model.pkl')
print("Model written to f1_model.pkl")